## Notebook to benchmark Trained NN VS Classifiers

In [ ]:
import torch
import os
import joblib
import numpy as np
import json
import time
from sklearn.metrics import (
    accuracy_score, recall_score, f1_score, precision_score, roc_auc_score,
    top_k_accuracy_score
)

from ensembler import EnsemblerClassifier, FinalWeightGeneratorNN 
from classifiers import (
    SVMClassifier, RBFClassifier, RandomForestClassifier, NaiveBayesClassifier, 
    LogisticRegressionClassifier, LDAClassifier, KNNClassifier, DecisionTreeClassifier,
    AdaBoostClassifier, GBMClassifier, XGBoostClassifier
)

# --- Configuration ---
CLIP_FEATURES_DIR = "clip_features"
VAL_FILE = os.path.join(CLIP_FEATURES_DIR, "val_features.pt")
SCALER_FILE = "scaler_model.joblib"
PCA_FILE = "pca_model.joblib"
LIME_FILE = "top_k_lime_indices.joblib"
NN_MODEL_PATH = "final_weight_generator.pth"
ENSEMBLE_RESULTS_FILE = "benchmark_results/ensemble_results.json"
OUTPUT_FILE = "benchmark_results/final_nn_one_hot_benchmark_results.json"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)



# --- Runtime Variables ---
MAX_TIME = 21.121998  # Maximum inference time observed in previous runs

In [2]:
print("Loading data and preprocessing models...")

# Load Features
data = torch.load(VAL_FILE)
val_img_features = data["image_features"]
val_txt_features = data["text_features"]
val_labels = data["labels"]

# Combine and Flatten
X_val = torch.cat((val_img_features, val_txt_features), dim=1)
X_val = X_val.view(X_val.size(0), -1).numpy()
y_val = val_labels.numpy()

# Load Preprocessors
scaler = joblib.load(SCALER_FILE)
pca = joblib.load(PCA_FILE)
lime = joblib.load(LIME_FILE)

# Transform Data
X_val_scaled = scaler.transform(X_val)
X_val_pca = pca.transform(X_val_scaled)

# Define Test Set
X_TEST = X_val_pca
Y_TRUE = y_val

print(f"Data Loaded. X_TEST shape: {X_TEST.shape}")

Loading data and preprocessing models...
Data Loaded. X_TEST shape: (1985, 563)


In [3]:
# --- Load Classifiers ---

classifier_names = [
    "SVMClassifier", "RBFClassifier", "RandomForestClassifier",
    "NaiveBayesClassifier", "LogisticRegressionClassifier",
    "LDAClassifier", "KNNClassifier", "DecisionTreeClassifier",
    "AdaBoostClassifier", "GBMClassifier", "XGBoostClassifier"
]

# Instantiate classifiers
classifiers = [
    SVMClassifier(), RBFClassifier(), RandomForestClassifier(), NaiveBayesClassifier(),
    LogisticRegressionClassifier(), LDAClassifier(), KNNClassifier(),
    DecisionTreeClassifier(), AdaBoostClassifier(), GBMClassifier(),
    XGBoostClassifier()
]

for clf in classifiers:
    clf.load()

Loaded model from: models_pca/SVM.joblib
Loaded label encoder from: models_pca/label_encoder.joblib
Loaded model from: models_pca/RBF.joblib
Loaded label encoder from: models_pca/label_encoder.joblib
Loaded model from: models_pca/RandomForest.joblib
Loaded label encoder from: models_pca/label_encoder.joblib
Loaded model from: models_pca/NaiveBayes.joblib
Loaded label encoder from: models_pca/label_encoder.joblib
Loaded model from: models_pca/LogisticRegression.joblib
Loaded label encoder from: models_pca/label_encoder.joblib
Loaded model from: models_pca/LDA.joblib
Loaded label encoder from: models_pca/label_encoder.joblib
Loaded model from: models_pca/KNN.joblib
Loaded label encoder from: models_pca/label_encoder.joblib
Loaded model from: models_pca/DecisionTree.joblib
Loaded label encoder from: models_pca/label_encoder.joblib
Loaded model from: models_pca/AdaBoost.joblib
Loaded label encoder from: models_pca/label_encoder.joblib
Loaded model from: models_pca/GBM.joblib
Loaded label e

In [4]:
from ensembler import FinalWeightGeneratorNN

#load model
model_path = "final_weight_generator.pth"
if os.path.exists(model_path):
    model = FinalWeightGeneratorNN()
    model.load_state_dict(torch.load(model_path, map_location=DEVICE))
    model.to(DEVICE)
    print("Model loaded successfully.")

Model loaded successfully.


In [ ]:
def compute_metrics(y_true, y_pred, y_proba, inference_time):
    """Computes a set of performance metrics."""
    
    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "recall": recall_score(y_true, y_pred, average='weighted', zero_division=0),
        "f1_score": f1_score(y_true, y_pred, average='weighted', zero_division=0),
        "precision": precision_score(y_true, y_pred, average='weighted', zero_division=0),
        "inference_time": inference_time
    }

    # AUC and Top-K require probability estimates
    if y_proba is not None and y_proba.ndim == 2:
        try:
            metrics["roc_auc"] = roc_auc_score(y_true, y_proba, multi_class='ovr', average='weighted', labels=np.arange(1, 91))
        except ValueError:
            metrics["roc_auc"] = 0.0 # Handle case where classes might be missing
        
        # Determine k for Top-K (assuming k=5 is appropriate for the number of classes)
        k_val = min(5, y_proba.shape[1])
        if k_val > 1:
            metrics["topk_accuracy"] = top_k_accuracy_score(y_true, y_proba, k=k_val, labels=np.arange(1, 91))
        else:
            metrics["topk_accuracy"] = metrics["accuracy"]

    return metrics

# Function to calculate the final weighted score based on random input parameters
def calculate_weighted_score(metrics_dict, weights):
    """
    Calculates a single weighted score for an ensemble based on the random input weights.
    The 7 weights apply to the following 7 metrics.
    """
    N_WEIGHTED_METRICS = len(weights)
    weighted_metric_keys = ["accuracy", "f1_score", "precision", "recall", "roc_auc", "topk_accuracy", "inference_time"]
    weighted_metric_keys = weighted_metric_keys[:N_WEIGHTED_METRICS]

    metric_values = []
    for key in weighted_metric_keys:
        val = metrics_dict.get(key)
        if val is None:
            val = 0.0
            if key == "inference_time": 
                val = 1.0 
        if key == "inference_time":
            val = 1 - (val / MAX_TIME)        
        metric_values.append(val)
    
    # Simple dot product for weighted sum/average of metrics
    weighted_sum = np.dot(np.array(metric_values), weights)
    
    return float(weighted_sum)

def get_base_metrics_tensor():
    """Loads base classifier metrics to serve as context for the NN."""
    if not os.path.exists(ENSEMBLE_RESULTS_FILE):
        raise FileNotFoundError(f"Missing {ENSEMBLE_RESULTS_FILE}. Cannot construct input tensor.")

    with open(ENSEMBLE_RESULTS_FILE, 'r') as file:
        data = json.load(file)

    metrics_list = []
    # Order must match NN training
    metric_keys = ["accuracy", "recall", "f1_score", "precision", "roc_auc", "top5_accuracy", "inference_time"]
    
    for entry in data.values():
        for key in metric_keys:
            metrics_list.append(entry.get(key, 0.0))
            
    return torch.tensor(metrics_list, dtype=torch.float32, device=DEVICE).unsqueeze(0)


In [6]:
context_tensor = get_base_metrics_tensor()

# Define Inputs
metric_names = ["accuracy", "f1_score", "precision", "recall", "roc_auc", "topk_accuracy", "inference_time"]
num_metrics = len(metric_names)

# Create Identity Matrix (7x7) for One-Hot vectors
one_hot_inputs = np.eye(num_metrics, dtype=np.float32)

results_storage = []

print(f"--- Starting One-Hot Benchmarking for {num_metrics} metrics ---\n")

for i in range(num_metrics):
    target_metric = metric_names[i]
    input_vector = one_hot_inputs[i]
    
    print(f"Testing optimization for: {target_metric.upper()} (Input: {input_vector})")

    # 1. Prepare NN Input [Weights + Context]
    input_params = torch.tensor(input_vector, device=DEVICE).unsqueeze(0) 
    full_input = torch.cat((input_params, context_tensor), dim=1)

    # 2. Get NN Prediction
    start_gen = time.time()
    with torch.no_grad():
        activation, weights_raw = model(full_input)
    gen_time = time.time() - start_gen

    # 3. Decode Output
    activation_bits = activation.detach().cpu().numpy().flatten()
    weight_values = weights_raw.detach().cpu().numpy().flatten()

    # Thresholding > 0.5
    active_indices = np.where(activation_bits > 0.5)[0]
    if len(active_indices) == 0:
        active_indices = [np.argmax(activation_bits)] # Fallback

    selected_clfs = [classifiers[idx] for idx in active_indices]
    selected_raw_weights = [weight_values[idx] for idx in active_indices]

    # Normalize weights
    final_weights = np.array(selected_raw_weights)
    if final_weights.sum() > 0:
        final_weights /= final_weights.sum()
    else:
        final_weights = np.ones_like(final_weights) / len(final_weights)

    clf_names = [type(c).__name__ for c in selected_clfs]
    print(f"  > Selected {len(clf_names)} classifiers: {clf_names}")

    # 4. Build Ensemble & Inference
    ensemble = EnsemblerClassifier(list(zip(selected_clfs, final_weights)))

    start_inf = time.time()
    y_pred = ensemble.classify(X_TEST)
    y_proba = ensemble.classify_proba(X_TEST)
    inf_time = time.time() - start_inf

    # 5. Compute Metrics
    current_metrics = compute_metrics(Y_TRUE, y_pred, y_proba, inf_time)
    
    print(f"  > Achieved {target_metric}: {current_metrics.get(target_metric, 0):.4f}\n")

    # 6. Store Data
    results_storage.append({
        "target_metric": target_metric,
        "input_vector": input_vector.tolist(),
        "selected_classifiers": clf_names,
        "selected_weights": final_weights.tolist(),
        "benchmark_metrics": current_metrics,
        "nn_generation_time": gen_time
    })

--- Starting One-Hot Benchmarking for 7 metrics ---

Testing optimization for: ACCURACY (Input: [1. 0. 0. 0. 0. 0. 0.])
  > Selected 3 classifiers: ['RBFClassifier', 'NaiveBayesClassifier', 'LogisticRegressionClassifier']


/home/tiago/thesis/venv/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/tiago/thesis/venv/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/tiago/thesis/venv/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/tiago/thesis/venv/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/tiago/thesis/venv/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defi

  > Achieved accuracy: 0.6408

Testing optimization for: F1_SCORE (Input: [0. 1. 0. 0. 0. 0. 0.])
  > Selected 5 classifiers: ['SVMClassifier', 'RBFClassifier', 'RandomForestClassifierWrapper', 'NaiveBayesClassifier', 'AdaBoostClassifierWrapper']


/home/tiago/thesis/venv/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/tiago/thesis/venv/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/tiago/thesis/venv/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/tiago/thesis/venv/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/tiago/thesis/venv/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defi

  > Achieved f1_score: 0.6096

Testing optimization for: PRECISION (Input: [0. 0. 1. 0. 0. 0. 0.])
  > Selected 4 classifiers: ['SVMClassifier', 'NaiveBayesClassifier', 'LogisticRegressionClassifier', 'GBMClassifier']


...

/home/tiago/thesis/venv/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/tiago/thesis/venv/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/tiago/thesis/venv/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/tiago/thesis/venv/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/tiago/thesis/venv/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defi

  > Achieved recall: 0.6363

Testing optimization for: ROC_AUC (Input: [0. 0. 0. 0. 1. 0. 0.])
  > Selected 3 classifiers: ['RBFClassifier', 'RandomForestClassifierWrapper', 'DecisionTreeClassifierWrapper']


/home/tiago/thesis/venv/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/tiago/thesis/venv/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/tiago/thesis/venv/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/tiago/thesis/venv/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/tiago/thesis/venv/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defi

  > Achieved roc_auc: 0.9285

Testing optimization for: TOPK_ACCURACY (Input: [0. 0. 0. 0. 0. 1. 0.])
  > Selected 3 classifiers: ['SVMClassifier', 'NaiveBayesClassifier', 'AdaBoostClassifierWrapper']
  > Achieved topk_accuracy: 0.9128

Testing optimization for: INFERENCE_TIME (Input: [0. 0. 0. 0. 0. 0. 1.])
  > Selected 1 classifiers: ['GBMClassifier']
  > Achieved inference_time: 0.0950



/home/tiago/thesis/venv/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/tiago/thesis/venv/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/tiago/thesis/venv/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/tiago/thesis/venv/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/tiago/thesis/venv/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defi

In [7]:
with open(OUTPUT_FILE, 'w') as f:
    json.dump(results_storage, f, indent=4)

print(f"✅ Benchmarking complete. Results saved to {OUTPUT_FILE}")

✅ Benchmarking complete. Results saved to yusuf_one_hot_benchmark_results.json
